Assembly of global flux jacobian matrix
- viscous flux jacobian matrix calculated by finite differencing method
- inviscid flux jacobian matrix calculated by analytical method
- boundary flux jacobian --> change to boundary_flux_jacobians_fd,  compatible with viscous_flux_jacobians_fd

In [5]:
import numpy as np
import sympy as sp
from pyau3d.utils import PltFileUtils, GrpFileUtils, UnkFileUtils
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
import pandas as pd

from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigs
from scipy.io import mmwrite, mmread
from scipy.sparse import save_npz, load_npz
from scipy.sparse import csr_matrix

import time
%matplotlib widget

In [6]:
# REQUIRED FUNCTIONS

# inviscid flux jacobian

def compute_dFdU(U, normal, gamma=1.4):
    """
    Compute the inviscid flux Jacobian matrix for 3D Euler equations.

    Based on equation (A.47) from Chung's Computationa Fluid Dynamics.
    Parameters
    ----------
    U : array-like, shape (5,)
        Conservative variable vector [rho, rho*u, rho*v, rho*w, rho*E]
        where:
        - rho: density
        - u, v, w: velocity components
        - E: specific total energy

    normal : array-like, shape (3,)
        Unit normal vector [n_x, n_y, n_z]

    gamma : float, optional
        Specific heat ratio (default: 1.4 for air)

    Returns
    -------
    dF_dU : ndarray, shape (5, 5)
        Flux Jacobian matrix
    """

    # Extract conservative variables
    rho = U[0]
    rho_u = U[1]
    rho_v = U[2]
    rho_w = U[3]
    rho_E = U[4]

    # Extract normal vector components
    n_x = normal[0]
    n_y = normal[1]
    n_z = normal[2]

    # Compute primitive variables
    u = rho_u / rho
    v = rho_v / rho
    w = rho_w / rho
    E = rho_E / rho

    # Compute auxiliary quantities (from A.48)
    # phi = 0.5 * (gamma - 1) * (u^2 + v^2 + w^2)
    phi = 0.5 * (gamma - 1.0) * (u**2 + v**2 + w**2)

    # V = n_x*u + n_y*v + n_z*w (normal component of velocity)
    V = n_x * u + n_y * v + n_z * w

    # a1 = gamma*E - phi
    a1 = gamma * E - phi

    # a2 = gamma - 1
    a2 = gamma - 1.0

    # a3 = gamma - 2
    a3 = gamma - 2.0

    # Initialize Jacobian matrix
    dF_dU = np.zeros((5, 5))

    # Row 1: d(rho*V)/d(U)
    dF_dU[0, 0] = 0
    dF_dU[0, 1] = n_x
    dF_dU[0, 2] = n_y
    dF_dU[0, 3] = n_z
    dF_dU[0, 4] = 0.0

    # Row 2: d(rho*u*V)/d(U)
    dF_dU[1, 0] = n_x * phi - u * V
    dF_dU[1, 1] = V - a3 * n_x * u
    dF_dU[1, 2] = n_y * u - a2 * n_x *v
    dF_dU[1, 3] = n_z * u - a2 * n_x * w
    dF_dU[1, 4] = a2 * n_x

    # Row 3: d(rho*v*V)/d(U)
    dF_dU[2, 0] = n_y * phi - v * V
    dF_dU[2, 1] = n_x * v - a2 * n_y *u
    dF_dU[2, 2] = V - a3 * n_y * v
    dF_dU[2, 3] = n_z * v - a2 * n_y * w
    dF_dU[2, 4] = a2 * n_y

    # Row 4: d(rho*w*V)/d(U)
    dF_dU[3, 0] = n_z * phi - w * V
    dF_dU[3, 1] = n_x * w - a2 * n_z *u
    dF_dU[3, 2] = n_y * w - a2 * n_z *v
    dF_dU[3, 3] =  V - a3 * n_z * w
    dF_dU[3, 4] = a2 * n_z

    # Row 5: d(rho*H*V)/d(U)
    dF_dU[4, 0] = V * (phi - a1)
    dF_dU[4, 1] = a1 * n_x - a2 * u * V
    dF_dU[4, 2] = a1 * n_y - a2 * v * V
    dF_dU[4, 3] = a1 * n_z - a2 * w * V
    dF_dU[4, 4] = gamma * V

    return dF_dU

def absolute_normal_jacobian(U, n, gamma=1.4):
    """
    Compute the absolute value of the normal Jacobian |A_n| for the
    3D Euler equations without tangent-vector ambiguity (eqs. 3.6.16–3.6.26).

    Parameters
    ----------
    U : array-like, shape (5,)
        Conservative variables [rho, rho*u, rho*v, rho*w, rho*E].
    n : array-like, shape (3,)
        Face normal vector (need not be unit length; normalised internally).
    gamma : float
        Ratio of specific heats (default 1.4).

    Returns
    -------
    abs_An : ndarray, shape (5, 5)
        Absolute normal Jacobian matrix.
    """
    U = np.asarray(U, dtype=float)
    n = np.asarray(n, dtype=float)
    n = n / np.linalg.norm(n)          # unit normal

    # ------------------------------------------------------------------ #
    # Primitive / thermodynamic quantities
    # ------------------------------------------------------------------ #
    rho  = U[0]
    vel  = U[1:4] / rho                # velocity vector  v = (u, v, w)
    E    = U[4]   / rho                # total energy per unit mass
    q2   = np.dot(vel, vel)            # |v|^2
    p    = (gamma - 1.0) * rho * (E - 0.5 * q2)
    c    = np.sqrt(gamma * p / rho)    # speed of sound
    H    = E + p / rho                 # total enthalpy per unit mass
    qn   = np.dot(vel, n)              # normal velocity  q_n
    M2   = q2 / c**2                   # Mach^2
    Mn   = qn / c                      # normal Mach number  M_n
    g1   = gamma - 1.0

    # ------------------------------------------------------------------ #
    # Contribution from eigenvalue qn  (eq. 3.6.24)
    # |qn| * (r2*l2' + r4*l4' + r5*l5')
    # ------------------------------------------------------------------ #
    mid = np.zeros((5, 5))

    # row 0
    mid[0, 0]   =  1.0 - 0.5 * g1 * M2
    mid[0, 1:4] =  (g1 / c**2) * vel
    mid[0, 4]   = -(g1 / c**2)

    # rows 1-3
    mid[1:4, 0]   = -0.5 * g1 * M2 * vel + qn * n
    mid[1:4, 1:4] =  (g1 / c**2) * np.outer(vel, vel) + np.eye(3) - np.outer(n, n)
    mid[1:4, 4]   = -(g1 / c**2) * vel

    # row 4
    mid[4, 0]   =  qn**2 - 0.5 * q2 * (1.0 + 0.5 * g1 * M2)
    mid[4, 1:4] =  (1.0 + 0.5 * g1 * M2) * vel - qn * n
    mid[4, 4]   = -0.5 * g1 * M2

    # ------------------------------------------------------------------ #
    # Contribution from eigenvalue (qn - c)  (eq. 3.6.25)
    # r1 * l1'
    # ------------------------------------------------------------------ #
    l1 = np.empty(5)
    l1[0]   =  0.25 * g1 * M2 + 0.5 * Mn
    l1[1:4] = -(g1 / (2.0 * c**2)) * vel - n / (2.0 * c)
    l1[4]   =  g1 / (2.0 * c**2)

    r1 = np.empty(5)
    r1[0]   =  1.0
    r1[1:4] =  vel - c * n
    r1[4]   =  H - qn * c

    A1 = np.outer(r1, l1)

    # ------------------------------------------------------------------ #
    # Contribution from eigenvalue (qn + c)  (eq. 3.6.26)
    # r3 * l3'
    # ------------------------------------------------------------------ #
    l3 = np.empty(5)
    l3[0]   =  0.25 * g1 * M2 - 0.5 * Mn
    l3[1:4] = -(g1 / (2.0 * c**2)) * vel + n / (2.0 * c)
    l3[4]   =  g1 / (2.0 * c**2)

    r3 = np.empty(5)
    r3[0]   =  1.0
    r3[1:4] =  vel + c * n
    r3[4]   =  H + qn * c

    A3 = np.outer(r3, l3)

    # ------------------------------------------------------------------ #
    # Assemble  |A_n| = |qn-c|*A1 + |qn|*mid + |qn+c|*A3   (eq. 3.6.19)
    # ------------------------------------------------------------------ #
    abs_An = abs(qn - c) * A1 + abs(qn) * mid + abs(qn + c) * A3

    return abs_An

def roe_average(UL, UR, gamma=1.4):
    """
    Compute the Roe-averaged conservative variable vector for the 3D Euler equations.
 
    Parameters
    ----------
    UL, UR : array-like, shape (5,)
        Left/right conservative vectors [rho, rho*u, rho*v, rho*w, rho*E].
    gamma : float
        Ratio of specific heats (default 1.4).
 
    Returns
    -------
    U_roe : ndarray, shape (5,)
        Roe-averaged conservative variable vector.
    """
    UL = np.asarray(UL, dtype=float)
    UR = np.asarray(UR, dtype=float)
 
    rhoL, rhoR = UL[0], UR[0]
    velL = UL[1:4] / rhoL
    velR = UR[1:4] / rhoR
    EL   = UL[4] / rhoL
    ER   = UR[4] / rhoR
 
    pL = (gamma - 1.0) * rhoL * (EL - 0.5 * np.dot(velL, velL))
    pR = (gamma - 1.0) * rhoR * (ER - 0.5 * np.dot(velR, velR))
    HL = EL + pL / rhoL    # total enthalpy per unit mass
    HR = ER + pR / rhoR
 
    wL = np.sqrt(rhoL)     # Roe weight for left state
    wR = np.sqrt(rhoR)     # Roe weight for right state
    ws = wL + wR
 
    rho_roe = wL * wR                          # = sqrt(rhoL * rhoR)
    vel_roe = (wL * velL + wR * velR) / ws
    H_roe   = (wL * HL   + wR * HR)   / ws
 
    # Recover E from H:  H = gamma*E - (gamma-1)/2 * |v|^2
    q2_roe = np.dot(vel_roe, vel_roe)
    E_roe  = (H_roe + (gamma - 1.0) * 0.5 * q2_roe) / gamma
 
    U_roe = np.empty(5)
    U_roe[0]   = rho_roe
    U_roe[1:4] = rho_roe * vel_roe
    U_roe[4]   = rho_roe * E_roe
    
    return U_roe

def inviscid_flux_jacobians(U_i, U_j, n,gamma=1.4):
    """
    Compute the normal inviscid flux Jacobian blocks at a cell face between
    cell i (left/owner) and cell j (right/neighbor).

    Parameters
    ----------
    U_i, U_j : array_like, shape (5,)
        Conservative variables [rho, rho*u, rho*v, rho*w, rho*E] of the
        two cells sharing the face.
    n : array_like, shape (3,)
        Unit normal vector [nx, ny, nz] of the face, pointing from i to j.
    gamma: float
        Ratio of specific heats
    Returns
    -------
    d(F)/d(U_i)  -- Inviscid Jacobian wrt the left-cell conservative state.
    d(F)/d(U_j)  -- Inviscid Jacobian wrt the right-cell conservative state.
    """

    U_roe = roe_average(U_i, U_j, gamma)
    abs_A_roe = absolute_normal_jacobian(U_roe, n, gamma=gamma)
    dFdUi = 0.5 * (compute_dFdU(U_i, n) + abs_A_roe)
    dFdUj = 0.5 * (compute_dFdU(U_j, n) - abs_A_roe)
    
    return dFdUi, dFdUj


# normal viscous flux jacobian

"""
3D Normal Viscous Flux Jacobian (interface between two control volumes)
========================================================================

Implements d(F_n^v)/dU_i  and  d(F_n^v)/dU_j, i.e. the Jacobian of the
normal (projected) viscous flux F_n^v with respect to the conservative
variables of the two cells (i = "left", j = "right") sharing a face,
following the construction:

        F_n^v = [0, -tau_nx, -tau_ny, -tau_nz, -tau_nn + q_n]^T      (eq. 4.12.5)

        d F_n^v / dU = (d F_n^v / dW) * (dW / dU)                    (eq. 4.12.15)

with W = [rho, u, v, w, p]^T.

Modeling choices (consistent with the reference derivation):
  * Interface (averaged) primitive states are the simple arithmetic
    mean of the two cells:           phi_avg = (phi_i + phi_j) / 2   (eq. 4.12.25)
  * The normal derivative of any quantity is the directional
    finite difference across the two cell centers:
            d(phi)/dn = (phi_j - phi_i) / ds                          (eq. 4.12.26)
  * Because only the normal derivative is available (no compact
    stencil for the tangential derivatives), the full gradient is
    approximated as  grad(phi) ~= (d phi/dn) * n   (i.e. only the
    derivative along n is retained -- the same simplification used
    to reduce tau_bar.n to a function of d()/dn only, eq. 4.12.12-14).
  * Viscosity follows Sutherland's law (eq. 4.4.4) and its dependence
    on rho and p (through T = p/(rho*R)) is differentiated exactly
    via the symbolic chain rule -- this reproduces eq. (4.12.20)-(4.12.24)
    without having to hand-transcribe each partial derivative.

The two Jacobian blocks (wrt U_i and wrt U_j) are exactly what an
implicit (e.g. Newton/line-implicit/LU-SGS) viscous-flux assembly
needs for the off-diagonal/diagonal contributions of a face.

Author: generated to accompany the "3D Normal Viscous Flux and Jacobian"
notes (section 4.12).
"""

# ----------------------------------------------------------------------
# Conservative -> primitive variables
# ----------------------------------------------------------------------
def cons_to_prim(U, gamma, R_gas):
    """
    U = [rho, rho*u, rho*v, rho*w, rho*E]
    returns rho, u, v, w, p, T  (numeric floats)
    """
    rho = U[0]
    u = U[1] / rho
    v = U[2] / rho
    w = U[3] / rho
    E = U[4] / rho
    p = (gamma - 1.0) * rho * (E - 0.5 * (u * u + v * v + w * w))
    T = p / (rho * R_gas)
    return rho, u, v, w, p, T


# ----------------------------------------------------------------------
# Sutherland's law (eq. 4.4.4), symbolic-friendly
# ----------------------------------------------------------------------
def sutherland_mu(T, mu0, T0, C):
    return mu0 * (T0 + C) / (T + C) * (T / T0) ** sp.Rational(3, 2)


# ----------------------------------------------------------------------
# dW/dU  (the right-hand matrix of eq. 4.12.15), evaluated at a given
# primitive state.  W = [rho, u, v, w, p]^T , U = [rho, rho u, rho v, rho w, rho E]^T
# ----------------------------------------------------------------------
def dWdU(rho, u, v, w, p, gamma):
    q2 = u * u + v * v + w * w
    M = np.array([
        [1.0,                 0.0,            0.0,            0.0,            0.0],
        [-u / rho,            1.0 / rho,       0.0,            0.0,            0.0],
        [-v / rho,            0.0,             1.0 / rho,      0.0,            0.0],
        [-w / rho,            0.0,             0.0,            1.0 / rho,      0.0],
        [0.5 * (gamma - 1.0) * q2, -(gamma - 1.0) * u, -(gamma - 1.0) * v, -(gamma - 1.0) * w, (gamma - 1.0)]
    ])
    return M


# ----------------------------------------------------------------------
# Symbolic construction of F_n^v as a function of the LEFT state
# (rho_L,u_L,v_L,w_L,p_L) and the RIGHT state (rho_R,u_R,v_R,w_R,p_R).
# Works for either state being symbolic; the other is plugged in as
# plain numbers.
# ----------------------------------------------------------------------
def _Fnv_symbolic(rho_L, u_L, v_L, w_L, p_L,
                   rho_R, u_R, v_R, w_R, p_R,
                   nx, ny, nz, ds,
                   gamma, R_gas, Pr, mu0, T0, Suth_C):

    T_L = p_L / (rho_L * R_gas)
    T_R = p_R / (rho_R * R_gas)

    # interface-averaged primitive state                       (eq. 4.12.25 style)
    rho_avg = (rho_L + rho_R) / 2
    u_avg   = (u_L   + u_R)   / 2
    v_avg   = (v_L   + v_R)   / 2
    w_avg   = (w_L   + w_R)   / 2
    T_avg   = (T_L   + T_R)   / 2

    mu_avg = sutherland_mu(T_avg, mu0, T0, Suth_C)             # mu(T_avg(rho,p)) -> exact chain rule via sympy

    # normal derivatives                                        (eq. 4.12.26)
    dudn = (u_R - u_L) / ds
    dvdn = (v_R - v_L) / ds
    dwdn = (w_R - w_L) / ds
    dTdn = (T_R - T_L) / ds

    # full gradient approximated by the normal derivative only:
    #   grad(phi) ~= (d phi/dn) * n   =>   d phi/dx_k = (d phi/dn) * n_k
    div = dudn * nx + dvdn * ny + dwdn * nz   # du/dx + dv/dy + dw/dz

    # Newtonian viscous stress tensor with Stokes' hypothesis      (eq. 4.12.16-19)
    tau_xx = mu_avg * (2 * dudn * nx - sp.Rational(2, 3) * div)
    tau_yy = mu_avg * (2 * dvdn * ny - sp.Rational(2, 3) * div)
    tau_zz = mu_avg * (2 * dwdn * nz - sp.Rational(2, 3) * div)
    tau_xy = mu_avg * (dudn * ny + dvdn * nx)
    tau_xz = mu_avg * (dudn * nz + dwdn * nx)
    tau_yz = mu_avg * (dvdn * nz + dwdn * ny)

    # projection along n                                          (eq. 4.12.6-4.12.9)
    tau_nx = tau_xx * nx + tau_xy * ny + tau_xz * nz
    tau_ny = tau_xy * nx + tau_yy * ny + tau_yz * nz
    tau_nz = tau_xz * nx + tau_yz * ny + tau_zz * nz
    tau_nn = tau_nx * u_avg + tau_ny * v_avg + tau_nz * w_avg

    # heat flux                                                    (eq. 4.12.11)
    kappa_avg = gamma * mu_avg / (Pr * (gamma - 1.0))
    q_n = -kappa_avg * dTdn

    F0 = sp.Integer(0)
    F1 = -tau_nx
    F2 = -tau_ny
    F3 = -tau_nz
    F4 = -tau_nn + q_n
    return sp.Matrix([F0, F1, F2, F3, F4])


# ----------------------------------------------------------------------
# Purely numeric evaluation of F_n^v(U_i, U_j, n, ds)  -- the discretized
# normal viscous flux function itself (no differentiation). This is the
# "flux(U)" used by the finite-difference Jacobian below, and it uses
# exactly the same modeling choices as the analytic version above:
#   * interface state = arithmetic mean of the two cells   (eq. 4.12.25)
#   * spatial (normal) derivative = directional finite difference
#         d(phi)/dn = (phi_j - phi_i) / ds                  (eq. 4.12.26)
#     i.e. the ONLY spatial derivative information available is the
#     one-sided difference of the two cell-center values along the
#     line joining them; the gradient is then assumed aligned with n:
#         grad(phi) ~= (d phi/dn) * n
#     (tangential derivatives are not reconstructed/are neglected).
# ----------------------------------------------------------------------
def Fnv_numeric(U_i, U_j, n, ds,
                 gamma=1.4, R_gas=287.0, Pr=0.72,
                 mu0=1.716e-5, T0=273.15, Suth_C=110.4):
    nx, ny, nz = n
    rho_L, u_L, v_L, w_L, p_L, T_L = cons_to_prim(U_i, gamma, R_gas)
    rho_R, u_R, v_R, w_R, p_R, T_R = cons_to_prim(U_j, gamma, R_gas)

    u_avg = (u_L + u_R) / 2
    v_avg = (v_L + v_R) / 2
    w_avg = (w_L + w_R) / 2
    T_avg = (T_L + T_R) / 2

    mu_avg = mu0 * (T0 + Suth_C) / (T_avg + Suth_C) * (T_avg / T0) ** 1.5

    dudn = (u_R - u_L) / ds
    dvdn = (v_R - v_L) / ds
    dwdn = (w_R - w_L) / ds
    dTdn = (T_R - T_L) / ds

    div = dudn * nx + dvdn * ny + dwdn * nz

    tau_xx = mu_avg * (2 * dudn * nx - (2.0 / 3.0) * div)
    tau_yy = mu_avg * (2 * dvdn * ny - (2.0 / 3.0) * div)
    tau_zz = mu_avg * (2 * dwdn * nz - (2.0 / 3.0) * div)
    tau_xy = mu_avg * (dudn * ny + dvdn * nx)
    tau_xz = mu_avg * (dudn * nz + dwdn * nx)
    tau_yz = mu_avg * (dvdn * nz + dwdn * ny)

    tau_nx = tau_xx * nx + tau_xy * ny + tau_xz * nz
    tau_ny = tau_xy * nx + tau_yy * ny + tau_yz * nz
    tau_nz = tau_xz * nx + tau_yz * ny + tau_zz * nz
    tau_nn = tau_nx * u_avg + tau_ny * v_avg + tau_nz * w_avg

    kappa_avg = gamma * mu_avg / (Pr * (gamma - 1.0))
    q_n = -kappa_avg * dTdn

    return np.array([0.0, -tau_nx, -tau_ny, -tau_nz, -tau_nn + q_n])


# ----------------------------------------------------------------------
# Finite-difference Jacobian (separate, independent function)
# ----------------------------------------------------------------------
def viscous_flux_jacobians_fd(U_i, U_j, n, ds, eps,
                               gamma=1.4, R_gas=287.0, Pr=0.72,
                               mu0=1.716e-5, T0=273.15, Suth_C=110.4):
    """
    Finite-difference estimate of d(F_n^v)/d(U_i) and d(F_n^v)/d(U_j),
    using the SAME discretized flux function F_n^v(U_i, U_j, n, ds) as
    the analytic version (Fnv_numeric above), but differentiating it
    by brute-force one-sided (forward) perturbation instead of exact
    symbolic/analytic differentiation.

    How the spatial derivative is determined
    -----------------------------------------
    The flux function F_n^v(U_i, U_j, n, ds) itself still needs a
    normal *spatial* gradient at the face. That gradient is built
    exactly as in the analytic implementation:
        d(phi)/dn = (phi_j - phi_i) / ds          (one-sided/directional
                                                    finite difference
                                                    across the two cell
                                                    centers, eq. 4.12.26)
        grad(phi) ~= (d phi/dn) * n                (tangential gradient
                                                    components are not
                                                    available/neglected;
                                                    eq. 4.12.12-14)
    This part is NOT what "eps" controls -- it is fixed by the mesh
    geometry (n, ds) and is identical to the analytic Jacobian's
    assumption.

    How the JACOBIAN (d Flux/d U) is determined
    ---------------------------------------------
    "eps" is a separate, independent perturbation used only to
    differentiate the flux function with respect to the conservative
    variables, via a forward finite difference, one component at a
    time:

        d F_n^v / dU_i[:,k]  ~=  ( F_n^v(U_i + eps*e_k, U_j, n, ds)
                                    - F_n^v(U_i, U_j, n, ds) ) / eps

        d F_n^v / dU_j[:,k]  ~=  ( F_n^v(U_i, U_j + eps*e_k, n, ds)
                                    - F_n^v(U_i, U_j, n, ds) ) / eps

    where e_k is the k-th unit vector in conservative-variable space
    (rho, rho*u, rho*v, rho*w, rho*E) and eps is a small, user-specified
    ABSOLUTE perturbation added directly to that conservative variable
    (same units as U_i[k]/U_j[k]). This is a forward difference, first
    order accurate in eps, i.e. error ~ O(eps); the other cell's state,
    n and ds are held fixed during each perturbation.

    Note: the formula is divided by eps (the perturbation), NOT by U;
    dividing by U would not produce a derivative -- it is eps that
    plays the role of "delta U" in (Flux(U+eps) - Flux(U)) / eps.

    Parameters
    ----------
    U_i, U_j : array_like, shape (5,)
        Conservative variables of the two cells sharing the face.
    n : array_like, shape (3,)
        Unit normal vector [nx, ny, nz], pointing from i to j.
    ds : float
        Distance between the two cell centers.
    eps : float
        User-specified absolute perturbation applied to each
        conservative-variable component in turn.
    gamma, R_gas, Pr, mu0, T0, Suth_C :
        Same gas/transport parameters as the analytic version.

    Returns
    -------
    J_i_fd : ndarray, shape (5,5)
        Finite-difference estimate of d(F_n^v)/d(U_i).
    J_j_fd : ndarray, shape (5,5)
        Finite-difference estimate of d(F_n^v)/d(U_j).
    """
    U_i = np.asarray(U_i, dtype=float)
    U_j = np.asarray(U_j, dtype=float)

    F_base = Fnv_numeric(U_i, U_j, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)

    J_i_fd = np.zeros((5, 5))
    J_j_fd = np.zeros((5, 5))

    for k in range(5):
        # --- perturb cell i, component k ---
        U_i_pert = U_i.copy()
        U_i_pert[k] += eps
        F_pert_i = Fnv_numeric(U_i_pert, U_j, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
        J_i_fd[:, k] = (F_pert_i - F_base) / eps

        # --- perturb cell j, component k ---
        U_j_pert = U_j.copy()
        U_j_pert[k] += eps
        F_pert_j = Fnv_numeric(U_i, U_j_pert, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
        J_j_fd[:, k] = (F_pert_j - F_base) / eps

    return J_i_fd, J_j_fd


# boundary flux jacobian

def riemann_invariant_bc(U_int, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas=287.0):
    """
    Ghost cell state using Riemann invariant (characteristic) boundary condition.

    Incoming/outgoing waves are split based on sign of eigenvalues along n.
    Outgoing invariants are taken from the interior; incoming from prescribed state.

    Parameters
    ----------
    U_int  : array (5,)   interior conservative state [rho, rho*u, rho*v, rho*w, rho*E]
    n      : array (3,)   outward unit normal
    gamma  : float        ratio of specific heats
    u_b, v_b, w_b : float prescribed velocity components at boundary
    T_b    : float        prescribed temperature at boundary
    P_b    : float        prescribed pressure at boundary
    R_gas  : float        specific gas constant (default: air, 287 J/kg/K)

    Returns
    -------
    U_ghost : ndarray (5,)  ghost cell conservative variables
    info    : dict          diagnostic info (flow regime, Riemann invariants)
    """
    U_int = np.asarray(U_int, dtype=float)
    n     = np.asarray(n,     dtype=float)
    n     = n / np.linalg.norm(n)

    cv = R_gas / (gamma - 1.0)

    # ------------------------------------------------------------------ #
    # Interior primitives
    # ------------------------------------------------------------------ #
    rho_i = U_int[0]
    vel_i = U_int[1:4] / rho_i
    E_i   = U_int[4] / rho_i                         # specific total energy
    Vn_i  = np.dot(vel_i, n)                          # normal velocity (signed)
    Vt_i  = vel_i - Vn_i * n                          # tangential velocity vector
    p_i   = (gamma - 1.0) * rho_i * (E_i - 0.5 * np.dot(vel_i, vel_i))
    T_i   = p_i / (rho_i * R_gas)
    c_i   = np.sqrt(gamma * R_gas * T_i)

    # ------------------------------------------------------------------ #
    # Prescribed (boundary) primitives
    # ------------------------------------------------------------------ #
    rho_b = P_b / (R_gas * T_b)
    vel_b = np.array([u_b, v_b, w_b])
    Vn_b  = np.dot(vel_b, n)
    Vt_b  = vel_b - Vn_b * n
    c_b   = np.sqrt(gamma * R_gas * T_b)

    # ------------------------------------------------------------------ #
    # Riemann invariants along normal direction
    #   R+ = Vn + 2c/(gamma-1)   associated with lambda = Vn + c
    #   R- = Vn - 2c/(gamma-1)   associated with lambda = Vn - c
    # ------------------------------------------------------------------ #
    fac = 2.0 / (gamma - 1.0)
    Rp_int = Vn_i + fac * c_i    # R+ from interior
    Rm_int = Vn_i - fac * c_i    # R- from interior
    Rp_b   = Vn_b + fac * c_b    # R+ from prescribed
    Rm_b   = Vn_b - fac * c_b    # R- from prescribed

    Mn_i = Vn_i / c_i            # normal Mach number (interior)

    # ------------------------------------------------------------------ #
    # Select invariants based on flow regime
    # ------------------------------------------------------------------ #
    if Mn_i <= -1.0:
        # Supersonic inflow: all characteristics incoming → use prescribed
        regime   = "supersonic_inflow"
        Vn_wall  = Vn_b
        c_wall   = c_b
        Vt_wall  = Vt_b
        s_wall   = P_b / (rho_b ** gamma)             # entropy from prescribed

    elif Mn_i >= 1.0:
        # Supersonic outflow: all characteristics outgoing → use interior
        regime   = "supersonic_outflow"
        Vn_wall  = Vn_i
        c_wall   = c_i
        Vt_wall  = Vt_i
        s_wall   = p_i / (rho_i ** gamma)

    elif Vn_i < 0.0:
        # Subsonic inflow: R+ outgoing (from interior), R- incoming (from prescribed)

        # R- incoming (from farfield to domain) --> use prescribed value from farfield/ inlet
        # tagential velocity and entropy use prescribed value from farfield
        # R+ from interior

        regime   = "subsonic_inflow"
        Rp_use   = Rp_int
        Rm_use   = Rm_b
        Vn_wall  = 0.5 * (Rp_use + Rm_use)
        c_wall   = 0.25 * (gamma - 1.0) * (Rp_use - Rm_use)
        Vt_wall  = Vt_b                               # tangential from prescribed
        s_wall   = P_b / (rho_b ** gamma)             # entropy from prescribed

    else:
        # Subsonic outflow: R- incoming (from prescribed), rest from interior
        # R- incoming (from farfield to domain) --> use prescribed value from farfield/ outlet
        # tagential velocity and entropy use prescribed value from interior/ domain
        # R+ from interior

        regime   = "subsonic_outflow"
        Rp_use   = Rp_int
        Rm_use   = Rm_b
        Vn_wall  = 0.5 * (Rp_use + Rm_use)
        c_wall   = 0.25 * (gamma - 1.0) * (Rp_use - Rm_use)
        Vt_wall  = Vt_i                               # tangential from interior
        s_wall   = p_i / (rho_i ** gamma)             # entropy from interior

    # ------------------------------------------------------------------ #
    # Reconstruct wall primitive state from (Vn, c, Vt, entropy)
    # ------------------------------------------------------------------ #
    if c_wall <= 0.0:
        raise ValueError(f"Non-physical c_wall={c_wall:.4f} in regime '{regime}'.")

    rho_wall = (c_wall**2 / (gamma * s_wall)) ** (1.0 / (gamma - 1.0))
    p_wall   = s_wall * rho_wall ** gamma
    T_wall   = p_wall / (rho_wall * R_gas)
    vel_wall = Vn_wall * n + Vt_wall

    # ------------------------------------------------------------------ #
    # Ghost cell: reflect wall state through interior so that
    # central average (U_int + U_ghost)/2 = U_wall
    # ------------------------------------------------------------------ #
    E_wall    = cv * T_wall + 0.5 * np.dot(vel_wall, vel_wall)
    U_wall_c  = np.array([
        rho_wall,
        rho_wall * vel_wall[0],
        rho_wall * vel_wall[1],
        rho_wall * vel_wall[2],
        rho_wall * E_wall,
    ])
    U_ghost = 2.0 * U_wall_c - U_int

    info = {
        "regime"  : regime,
        "Mn_i"    : Mn_i,
        "Rp_int"  : Rp_int, "Rm_int": Rm_int,
        "Rp_b"    : Rp_b,   "Rm_b"  : Rm_b,
        "Vn_wall" : Vn_wall, "c_wall": c_wall,
        "p_wall"  : p_wall,  "T_wall": T_wall,
    }
    return U_ghost, info

# ghost_state function with Riemann Invariant boundary conditions

def ghost_state(U, bc_type, n, gamma=1.4, R_gas=287.0,
                u_b=0.0, v_b=0.0, w_b=0.0, T_b=300.0, P_b=101325.0):
    """
    Compute ghost cell state for compressible N-S boundary conditions.

    Parameters
    ----------
    U       : array (5,)  conservative variables [rho, rho*u, rho*v, rho*w, rho*E]
    bc_type : str         'slip', 'noslip', or 'riemann'
    n       : array (3,)  outward unit normal

    For bc_type == 'riemann' only:
    gamma   : float       ratio of specific heats
    R_gas   : float       specific gas constant (J/kg/K)
    u_b, v_b, w_b : float prescribed boundary velocity
    T_b     : float       prescribed boundary temperature
    P_b     : float       prescribed boundary pressure

    Returns
    -------
    U_ghost : ndarray (5,)
    """
    U = np.asarray(U, dtype=float)
    n = np.asarray(n, dtype=float)
    n = n / np.linalg.norm(n)

    rho   = U[0]
    vel   = U[1:4] / rho
    rho_E = U[4]

    if bc_type == 'slip':
        vn        = np.dot(vel, n)
        vel_ghost = vel - 2.0 * vn * n
        return np.array([rho, *(rho * vel_ghost), rho_E])

    elif bc_type == 'noslip':
        return np.array([rho, *(-rho * vel), rho_E])

    elif bc_type == 'riemann':
        return riemann_invariant_bc(U, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas)[0]  # only return the first output, the second output is for debugging

    else:
        raise ValueError(f"Unknown bc_type '{bc_type}'. Use 'slip', 'noslip', or 'riemann'.")
    
def boundary_flux_jacobian_fd(U_i, bc_type, n, ds, eps,
                            gamma=1.4, R_gas=287.0, Pr=0.72,
                            mu0=1.716e-5, T0=273.15, Suth_C=110.4,
                            u_b=0.0, v_b=0.0, w_b=0.0, T_b=300.0, P_b=101325.0):
    """
    Compute the boundary flux Jacobian dF_bc/dU_i at a boundary face.

    The ghost cell U_g is constructed from the interior state U_i via
    ghost_state(), then the appropriate flux Jacobian is evaluated using
    (U_i, U_g) as the left/right pair. Only the left (interior) block
    dF/dU_i is returned; the ghost-cell dependence on U_i is frozen
    (standard first-order boundary Jacobian treatment).

    Parameters
    ----------
    U_i     : array (5,)    interior conservative state
    bc_type : str           'slip', 'riemann' --> inviscid Jacobian
                            'noslip'          --> viscous Jacobian (inviscid neglected)
    n       : array (3,)    outward unit normal
    ds      : float         cell-center to face distance (needed for 'noslip' only)
    eps     : float         pertubation for Finite Differencing of "no-slip" viscous flux jacobian
 
    For bc_type == 'riemann':
    u_b, v_b, w_b, T_b, P_b : float  prescribed boundary state

    Returns
    -------
    J_bc : ndarray (5, 5)   dF_bc / dU_i
    """
    U_i = np.asarray(U_i, dtype=float)
    n   = np.asarray(n,   dtype=float)
    n   = n / np.linalg.norm(n)

    # ------------------------------------------------------------------ #
    # Build ghost cell
    # ------------------------------------------------------------------ #
    if bc_type == 'riemann':
        U_g = ghost_state(U_i, 'riemann', n, gamma, R_gas,
                          u_b, v_b, w_b, T_b, P_b)
    else:
        U_g = ghost_state(U_i, bc_type, n)

    # ------------------------------------------------------------------ #
    # Select flux Jacobian
    # ------------------------------------------------------------------ #
    if bc_type in ('slip', 'riemann'):
        # inviscid (Roe-based) Jacobian; return only the interior block
        J_bc, _ = inviscid_flux_jacobians(U_i, U_g, n, gamma)

    elif bc_type == 'noslip':
        if ds == 1.0:
            import warnings
            warnings.warn("boundary_flux_jacobian: ds=1.0 (default) used for 'noslip' BC. "
                      "Set ds to the actual cell-to-wall distance. STH IS WRONG WITH BOUNDARY List FILE", stacklevel=2)
        
        # viscous Jacobian only (inviscid flux is zero at a no-slip wall)
        J_bc, _ = viscous_flux_jacobians_fd(U_i, U_g, n, ds, eps,
                                  gamma=gamma, R_gas=R_gas, Pr=Pr,
                                  mu0=mu0, T0=T0, Suth_C=Suth_C)
    else:
        raise ValueError(f"Unknown bc_type '{bc_type}'. "
                         "Use 'slip', 'noslip', or 'riemann'.")

    return J_bc


In [ ]:
# assemble_global_flux jacobian

"""
Global Flux Jacobian Assembly — fort.txt + boundary_list inputs

by FINITE DIFFERENCING of the viscous flux jacobian and boundary flux jacobian (for "no-slip" only)
================================================================
Inviscid (Euler) flux only, Riemann invariant boundary conditions.

Input formats
-------------
fortfile       ndarray (M, 5)   interior face table
                 col 0 : node i  (1-based, always < j)
                 col 1 : node j  (1-based)
                 col 2-4: [Ax, Ay, Az]  face area-weighted normal vector
                           norm = face area, direction = outward from i toward j

U_list         ndarray (N, 5)   conservative variables per cell
                 [rho, rho*u, rho*v, rho*w, rho*E]

coord          ndarray (N, 3)   cell-centre coordinates (not used for inviscid)

boundary_list  ndarray (B, 12) 
                col 0    : node i (1-based)
                col 1-3  : [Ax, Ay, Az]  outward area-weighted normal
                col 4    : ds
                col 5    : bc_type   0=riemann | 1=slip | 2=noslip
                col 6-8  : u_b, v_b, w_b
                col 9    : T_b
                col 10   : P_b
                col 11   : flag (surface id, for filtering/debugging)
eps             float         pertubation for Finite Differencing of "no-slip" boundary flux jacobian and viscous flux jacobian

Assumed to be appended after (or imported from) the file that defines:
  inviscid_flux_jacobians, boundary_flux_jacobian
"""

# =============================================================================
# ASSEMBLER
# =============================================================================
 
def assemble_global_jacobian_fd(
        fortfile, U_list, coord, boundary_list, eps,
        gamma=1.4, R_gas=287.0,
        viscous=False,
        Pr=0.72, mu0=1.716e-5, T0=273.15, Suth_C=110.4):
    """
    Assemble the global flux Jacobian from fort-file face data.
 
    Interior face loop  (fortfile)
    ──────────────────
    For face (i→j) with area-weighted normal A_ij and face area Area = |A_ij|:
 
        dF_total = dF_inviscid  [+ dF_viscous  if viscous=True]
 
        J[i,i] += dF_total_i * Area       J[i,j] += dF_total_j * Area
        J[j,i] -= dF_total_i * Area       J[j,j] -= dF_total_j * Area
 
    Viscous interior ds
    ───────────────────
    ds = |dot(coord[j] − coord[i], n̂)|   (projected cell-to-cell distance)
 
    Boundary face loop  (boundary_list)
    ──────────────────
    For boundary face on cell i with outward normal Ag_bc to neighbour ghost cell:
 
        J[i,i] += J_bc * Area
 
    For 'slip' and 'riemann' BCs:  J_bc is the inviscid boundary Jacobian.
    For 'noslip' BC:                J_bc is the viscous boundary Jacobian.
        ds is obtained from boundary_list[5]
 
    Parameters
    ----------
    fortfile      : ndarray (M, 5)  Note that first column nodes are 1-based index
    U_list        : ndarray (N, 5)   conservative variables (0-based index)
    coord         : ndarray (N, 3)   cell-centre coordinates (0-based index)
    boundary_list : ndarray (B, 11)  Note that first column nodes are 1-based index
    gamma         : float
    R_gas         : float   gas constant [J/(kg·K)]
    viscous       : bool    include viscous flux Jacobians (interior + noslip BC)
    Pr            : float   Prandtl number
    mu0           : float   reference dynamic viscosity [Pa·s]
    T0            : float   reference temperature for Sutherland [K]
    Suth_C        : float   Sutherland constant [K]
 
    Returns
    -------
    J : ndarray (5N, 5N)   dense global Jacobian
    """
    bc_str = {0: 'riemann', 1: 'slip', 2: 'noslip'}
    N      = U_list.shape[0]
    J      = lil_matrix((5 * N, 5 * N))
 
    visc_kw = dict(gamma=gamma, R_gas=R_gas, Pr=Pr,
                   mu0=mu0, T0=T0, Suth_C=Suth_C)
 
    # ── Interior faces ────────────────────────────────────────────────────────
    for row in fortfile:
        i    = int(row[0]) - 1                        # 0-based
        j    = int(row[1]) - 1
        A_ij = np.asarray(row[2:5], dtype=float)
        Area = np.linalg.norm(A_ij)
        if Area < 1e-14:
            continue
        n_ij = A_ij / Area
 
        # Inviscid contribution
        dFi, dFj = inviscid_flux_jacobians(U_list[i], U_list[j], n_ij, gamma)
 
        # Viscous contribution
        if viscous:
            ds   = abs(float(np.dot(coord[j] - coord[i], n_ij)))
            ds = max(ds, 1e-14) # interesting treatment
 
            dFi_v, dFj_v = viscous_flux_jacobians_fd(
                U_list[i], U_list[j], n_ij, ds, eps, **visc_kw) # use **visc_kw to key in arguments in dictionary visc_kw
            dFi = dFi + dFi_v # including both invisicd and viscous
            dFj = dFj + dFj_v
 
        si = slice(5 * i, 5 * i + 5)
        sj = slice(5 * j, 5 * j + 5)
        J[si, si] += dFi * Area
        J[si, sj] += dFj * Area
        J[sj, si] -= dFi * Area
        J[sj, sj] -= dFj * Area
 
    # ── Boundary faces ────────────────────────────────────────────────────────
    for row in boundary_list:
        i    = int(row[0]) - 1 # convert first column into 0-based index
        A_bc = np.asarray(row[1:4], dtype=float)
        Area = np.linalg.norm(A_bc)
        ds   = float(row[4])
        if Area < 1e-14:
            continue
        n_bc = A_bc / Area
        bc   = bc_str[int(row[5])]
        u_b, v_b, w_b, T_b, P_b = (float(row[6]), float(row[7]),
                                        float(row[8]), float(row[9]), float(row[10]))

        J_bc = boundary_flux_jacobian_fd(U_list[i], bc, n_bc, ds, eps, u_b=u_b, v_b=v_b, w_b=w_b, T_b=T_b, P_b=P_b,**visc_kw)
        si = slice(5 * i, 5 * i + 5)
        J[si, si] += J_bc * Area

    return J  # directly return lil matrix

In [8]:
# build_boundary_list function

# Area normal function - helper function

def area_normals(coord, ifac3=None, ifac4=None):
    """
    Compute area-weighted normal vectors (Ax, Ay, Az) at each nodes,
    accumulated from triangle and/or quadrilateral faces. No PolyData is
    built; only the (N, 3) array is returned.
 
    For each face, the area-normal vector is:
        triangle : 0.5 * cross(v1 - v0, v2 - v0)
        quad     : sum of the two triangle area-normals from splitting
                   the quad (0,1,2) + (0,2,3)
    Each face's area-normal is split equally among its vertices and summed.

    That is how it accounted from the node-centered formulation
 
    Args:
        coord : (N, 3) array of mesh point coordinates
        ifac3 : (M3, 3) array of triangle connectivity, zero-based (or None)
        ifac4 : (M4, 4) array of quad connectivity, zero-based (or None)
 
    Returns:
        anor : (N, 3) array of area-weighted normals (Ax, Ay, Az) per point
    """
    coord = np.asarray(coord, dtype=float)
    anor = np.zeros_like(coord)
 
    if ifac3 is not None and len(ifac3) > 0:
        ifac3 = np.asarray(ifac3)
        v0, v1, v2 = coord[ifac3[:, 0]], coord[ifac3[:, 1]], coord[ifac3[:, 2]]
        face_anor = 0.5 * np.cross(v1 - v0, v2 - v0)   # (M3, 3)
        contrib = -face_anor / 3.0                      # sign matches ref code
        for k in range(3):
            np.add.at(anor, ifac3[:, k], contrib)
 
    if ifac4 is not None and len(ifac4) > 0:
        ifac4 = np.asarray(ifac4)
        v0, v1, v2, v3 = (coord[ifac4[:, 0]], coord[ifac4[:, 1]],
                          coord[ifac4[:, 2]], coord[ifac4[:, 3]])
        n1 = 0.5 * np.cross(v1 - v0, v2 - v0)
        n2 = 0.5 * np.cross(v2 - v0, v3 - v0)
        face_anor = n1 + n2                              # (M4, 3)
        contrib = -face_anor / 4.0
        for k in range(4):
            np.add.at(anor, ifac4[:, k], contrib)
 
    return anor


def boundary_geometry(pltfile, coord, fortfile, flag):
    """
    Wall-normal distance for each boundary node on a given surface.

    ds_i = mean over interior neighbours j of | n_hat_i . (x_j - x_i) |

    Returns
    -------
    (Nb, 5) array: [node_id (0-based), Abx, Aby, Abz, ds]

    """
    # use extract_surface_real to ifac4 as well 
    surface_nodes, tri_connect, quad_connect = pltfile.extract_surface_real(flag = flag)

    # get surface coordinates (0 based indexing)
    surface_coord = coord[surface_nodes]

    surface_area_normals = area_normals(surface_coord, ifac3=tri_connect, ifac4 = quad_connect)

    n_hat = surface_area_normals / np.linalg.norm(surface_area_normals, axis=1, keepdims=True)

    # global node id -> row in surface_nodes / n_hat
    local_index = -np.ones(coord.shape[0], dtype=int)
    local_index[surface_nodes] = np.arange(len(surface_nodes))

    surface_mask = np.zeros(coord.shape[0], dtype=bool)
    surface_mask[surface_nodes] = True

    ds_lists = {}   # boundary node id -> list of projected distances to its interior neighbours

    for row in fortfile:
        a = int(row[0]) - 1   # 0-based
        b = int(row[1]) - 1

        a_surf = surface_mask[a]
        b_surf = surface_mask[b]

        if a_surf and not b_surf:
            node, neigh = a, b
        elif b_surf and not a_surf:
            node, neigh = b, a
        else:
            continue   # both on surface (tangential edge) or both interior -> not what we want

        normal = n_hat[local_index[node]]
        vec = coord[neigh] - coord[node]
        ds_proj = abs(np.dot(vec, normal))

        ds_lists.setdefault(node, []).append(ds_proj)

    node_ids = np.array(sorted(ds_lists.keys()))
    ds = np.array([np.mean(ds_lists[n]) for n in node_ids])

    return np.column_stack([node_ids, surface_area_normals, ds])


def build_boundary_list(pltfile, coord, fortfile, flags_bc_map):
    """
    boundary_list : (B, 12) ndarray
        col 0    : node i (1-based)
        col 1-3  : [Ax, Ay, Az]  outward area-weighted normal
        col 4    : ds
        col 5    : bc_type   0=riemann | 1=slip | 2=noslip
        col 6-8  : u_b, v_b, w_b
        col 9    : T_b
        col 10   : P_b
        col 11   : flag (surface id, for filtering/debugging)
    """
    rows = []
    for flag, bc in flags_bc_map.items():
        geom = boundary_geometry(pltfile, coord, fortfile, flag)
        n = geom.shape[0]
        bc_cols = np.tile([
            float(bc['bc_type']), float(bc['u_b']), float(bc['v_b']),
            float(bc['w_b']), float(bc['T_b']), float(bc['P_b']),
        ], (n, 1))
        node_1based = geom[:, [0]] + 1.0
        flag_col = np.full((n, 1), flag, dtype=float)
        rows.append(np.hstack([node_1based, geom[:, 1:5], bc_cols, flag_col]))
        
    return np.vstack(rows)

In [9]:
# utility functions

def specific_energy(rho, p, ux, uy, uz, gamma=1.4):
    return p / ((gamma - 1.0) * rho) + 0.5 * (ux**2 + uy**2 + uz**2)

def conservative_variables(rst, GAMMA):
    """Return conservative variables U on the surface."""
    E = specific_energy(rst.rho, rst.p, rst.ux, rst.uy, rst.uz, GAMMA)

    U = np.column_stack((
        rst.rho,
        rst.rho * rst.ux,
        rst.rho * rst.uy,
        rst.rho * rst.uz,
        rst.rho * E
    ))
    return U

Flux Jacobian Assembly - by FD of viscous flux jacobian
1. Read neccessary files
- rst
- plt
- fortfile

return fortfile, U_list, coord_list, boundary_list

2. implement assemble_global_jacobian_fd

- plot the jacobian over the domain to check the pattern , does it make sense?
- 3 cases: only inviscid, inviscid + viscous, inviscid + viscous + boundary flux
- note that it takes an extra argument eps

3. Sanity checks

a) 1D uniform inviscid flow
- with boundary conditions and without boundary conditions
- check eigenvalues 

b) 2D uniform inviscid flow
- structure and unstructured grid
- uniform flow 
- check eigenvalues 

4. Plot the flux jacobian matrix on the mesh

In [10]:
# 1. Read neccessary files
Mesh = 21228
Re   = 60
Mach = 0.2
gamma = 1.4
R_gas = 287.0          # confirm units match the solver

dir = f"./2d_cylinder_{Mesh}_Re{Re}_M{Mach}"
pltfile = PltFileUtils(f"{dir}/cylinder.plt")
rstfile = UnkFileUtils(f"{dir}/cylinder.rst", extend=False)

fortfile = pd.read_csv(f"{dir}/fort.864", sep=r'\s+', header=None).to_numpy()

rstfile._primitive()
U_list = conservative_variables(rstfile, gamma)
coord  = pltfile.coord

u_in  = 68.0525;  v_in  = 0.0;  w_in  = 0.0
T_in  = 288.15;   P_in  = 1.32702
u_out = 68.0525;  v_out = 0.0;  w_out = 0.0
T_out = 288.15;   P_out = 1.32702

# bc_type   0=riemann | 1=slip | 2=noslip
flags_bc_map = {
    1: {'bc_type': 0, 'u_b': u_in,  'v_b': v_in,  'w_b': w_in,  'T_b': T_in,  'P_b': P_in},
    2: {'bc_type': 0, 'u_b': u_out, 'v_b': v_out, 'w_b': w_out, 'T_b': T_out, 'P_b': P_out},
    3: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
    4: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
    5: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
    6: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
    7: {'bc_type': 2, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
}


boundary_list = build_boundary_list(pltfile, coord, fortfile, flags_bc_map)

In [13]:
eps = 1e-8

t0 = time.perf_counter()

global_jacobian_fd = assemble_global_jacobian_fd(fortfile, U_list, coord, boundary_list, eps, gamma = gamma, R_gas = R_gas, viscous =  True)

# save run time 
t = time.perf_counter() - t0

print(f"eps: {eps}")
print(f"Run time: {t}s")

eps: 1e-08
Run time: 262.24411559989676s


In [15]:
# save data

data_dir = "./data"
# mmwrite(f"{data_dir}/jacobian_cylinder_12625_Re{Re}_M{Mach}_fd.mtx", global_jacobian_fd)

# save to .npz format with CSR (more efficient )
save_npz(f"{data_dir}/jacobian_cylinder_{Mesh}_Re{Re}_M{Mach}_fd.npz", csr_matrix(global_jacobian_fd))